# Analiza Open Library skupa podataka

Skup podataka preuzet je sa projekta Open Library (Internet Archive) u obliku mesecnih dump fajlova.
Koriste se tri kolekcije:

- `works` - knjizevna dela na konceptualnom nivou (naslov, autori, teme, godina objavljivanja)
- `authors` - podaci o autorima (ime, datum rodjenja i smrti, biografija)
- `ratings` - korisnicke ocene dela (vrednost od 1 do 5)

Cilj analize je odgovaranje na deset pitanja postavljenih iz perspektive razlicitih uloga,
nakon cega sledi kreiranje indeksa za optimizaciju upita.

## Priprema okruzenja

Povezivanje sa lokalnom MongoDB instancom i definisanje kolekcija.

In [ ]:
from pymongo import MongoClient, ASCENDING, DESCENDING
import json
import time

client = MongoClient("mongodb://localhost:27017/")
db = client["open_library"]

works = db["works"]
authors = db["authors"]
ratings = db["ratings"]

## Ucitavanje podataka

Dump fajlovi za `works` i `authors` su tab-separated, gde svaki red ima pet kolona:
tip zapisa, kljuc, revizija, datum izmene i JSON dokument. Od interesa je peta kolona.

Fajlovi su veliki (works 23 GB, authors 6 GB) pa se citaju red po red bez ucitavanja u memoriju.
Ucitavanje se vrsi u serijama radi brzine, a neispravni JSON redovi se preskacu uz brojanje.

In [ ]:
def ucitaj_dump(putanja, kolekcija, ocekivani_tip, batch_size=1000):
    kolekcija.drop()
    batch = []
    ubaceno = 0
    preskoceno = 0
    with open(putanja, "r", encoding="utf-8") as f:
        for i, linija in enumerate(f):
            delovi = linija.split("\t")
            if len(delovi) < 5:
                preskoceno += 1
                continue
            if delovi[0] != ocekivani_tip:
                continue
            try:
                dokument = json.loads(delovi[4])
            except json.JSONDecodeError:
                preskoceno += 1
                continue
            batch.append(dokument)
            if len(batch) >= batch_size:
                kolekcija.insert_many(batch)
                ubaceno += len(batch)
                batch = []
            if (i + 1) % 100000 == 0:
                print(f"  obradjeno redova: {i + 1:,} | ubaceno: {ubaceno:,}")
    if batch:
        kolekcija.insert_many(batch)
        ubaceno += len(batch)
    print(f"Zavrseno. Ubaceno: {ubaceno:,} | preskoceno: {preskoceno:,}")
    return ubaceno

In [ ]:
print("Ucitavanje works kolekcije...")
ucitaj_dump("ol_dump_works_2026-05-31.txt", works, "/type/work")

In [ ]:
print("Ucitavanje authors kolekcije...")
ucitaj_dump("ol_dump_authors_2026-05-31.txt", authors, "/type/author")

In [ ]:
def ucitaj_ratings(putanja, kolekcija, batch_size=1000):
    kolekcija.drop()
    batch = []
    ubaceno = 0
    preskoceno = 0
    with open(putanja, "r", encoding="utf-8") as f:
        for linija in f:
            delovi = linija.rstrip("\n").split("\t")
            if len(delovi) < 4:
                preskoceno += 1
                continue
            try:
                dokument = {
                    "work_key": delovi[0],
                    "rating": int(delovi[2]),
                    "date": delovi[3]
                }
            except ValueError:
                preskoceno += 1
                continue
            batch.append(dokument)
            if len(batch) >= batch_size:
                kolekcija.insert_many(batch)
                ubaceno += len(batch)
                batch = []
    if batch:
        kolekcija.insert_many(batch)
        ubaceno += len(batch)
    print(f"Zavrseno. Ubaceno: {ubaceno:,} | preskoceno: {preskoceno:,}")
    return ubaceno

In [ ]:
print("Ucitavanje ratings kolekcije...")
ucitaj_ratings("ol_dump_ratings_2026-05-31.txt", ratings)

In [ ]:
print("works  :", works.count_documents({}))
print("authors:", authors.count_documents({}))
print("ratings:", ratings.count_documents({}))